In [ ]:
from pathlib import Path
from collections import defaultdict
from eppy.modeleditor import IDF

# --- Load IDFs ---
IDF.setiddname("/usr/local/EnergyPlus-9-5-0/Energy+.idd")

original = IDF("/workspaces/CUBES/beizaee_validation/runs_h28_test_validation/beizaee/conventional_control__part_0_2__eff_quadratic/modified_test.idf")
new      = IDF("/workspaces/CUBES/beizaee/experiments/thermostat/runs_validation/beizaee/conventional_control__part_2__eff_quadratic/modified_test.idf")

objs1 = original.idfobjects
objs2 = new.idfobjects

types1 = set(objs1.keys())
types2 = set(objs2.keys())

print("\n=== Object types only in ORIGINAL ===")
for t in sorted(types1 - types2):
    print(" ", t)

print("\n=== Object types only in NEW ===")
for t in sorted(types2 - types1):
    print(" ", t)

print("\n=== Object count differences ===")
for t in sorted(types1 & types2):
    if len(objs1[t]) != len(objs2[t]):
        print(f"{t}: ORIGINAL={len(objs1[t])}   NEW={len(objs2[t])}")

# --- helpers ---
TOL = 1e-6  # ignore small float differences

def clean(v):
    if isinstance(v, str):
        return v.strip().lower()
    return v

def by_name(objects):
    out = {}
    for obj in objects:
        name = getattr(obj, "Name", None)
        if name:
            out[name.lower()] = obj
    return out

print("\n=== Field-level differences (matched by NAME) ===")

# --- compare field-by-field ---
for t in sorted(types1 & types2):
    map1 = by_name(objs1[t])
    map2 = by_name(objs2[t])

    names1 = set(map1.keys())
    names2 = set(map2.keys())
    common = names1 & names2

    only_in_original = names1 - names2
    only_in_new      = names2 - names1

    if only_in_original:
        print(f"\n[{t}] Present only in ORIGINAL:")
        for n in sorted(only_in_original):
            print("  ", n)

    if only_in_new:
        print(f"\n[{t}] Present only in NEW:")
        for n in sorted(only_in_new):
            print("  ", n)

    for name in sorted(common):
        o1 = map1[name]
        o2 = map2[name]

        maxfields = min(len(o1.obj), len(o2.obj))

        for field in range(maxfields):
            v1 = clean(o1.obj[field])
            v2 = clean(o2.obj[field])
            label = o1.objls[field]

            # --- ignore tiny float differences ---
            if isinstance(v1, (int, float)) and isinstance(v2, (int, float)):
                if abs(v1 - v2) < TOL:
                    continue
            else:
                if v1 == v2:
                    continue

            # real difference
            print(f"\n[{t}] {name}")
            print(f"  {label}: ORIGINAL={v1}   NEW={v2}")
            break
